1번째(모든 코드),코랩 서버 연결(cpu)

In [ ]:
#2번째(모든 코드),Drive 연결

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#3번쨰(모든 코드) 3.11 가상환경 만들기 (방법 B) 
%pip install -q uv

!uv venv /content/py311 --python 3.11.17
!uv pip install -p /content/py311/bin/python -r /content/drive/MyDrive/port-ai/env/requirements-py311.txt

#버젼 확인 
!/content/py311/bin/python -c "import sys; print(sys.version)"

#패키지 목록
!uv pip list -p /content/py311/bin/python

%pip install -q -r /content/drive/MyDrive/port-ai/env/requirements-notebook.txt

In [ ]:
from pathlib import Path
import sys
%pip install -q -r /content/drive/MyDrive/port-ai/env/requirements-notebook.txt
BASE = Path('/content/drive/MyDrive/port-ai')   # 절대 경로는 이 한 줄만 설정
if str(BASE) not in sys.path:  
    sys.path.insert(0,str(BASE))                #경로가 없으면 만들기 


DATA, OUT = BASE / 'open', BASE / 'outputs'     


from src.normalize import normalize_name,ko_phon,eng_phon


print(normalize_name("m/v Morning-Courie")) #접두사와 하이픈이 정리된다 

print(normalize_name("제7하나")) #접두사와 하이픈이 정리된다 
print(normalize_name("No. 7 Hana")) #접두사와 하이픈이 정리된다 
print(normalize_name("No. 7 Hana")) #접두사와 하이픈이 정리된다 

for i in normalize_name("제7하나"):
    print(f"코폰  + {ko_phon(i)}")
    

In [ ]:
from calendar import c
import re

import pandas as pd
from polars import Catalog
from src.normalize import normalize_name,ko_phon,eng_phon
from src.config import train_col


c_df = pd.read_csv(DATA/'vessel_catalog.csv')

t_df = pd.read_csv(DATA/'train.csv')

catalog_vessel_name_list = []

#후보 데이터 
for i in c_df[train_col['VESSEL_NAME']]:
    name = "".join(normalize_name(i))
    catalog_vessel_name_list.append(eng_phon(name))

#vessel_name_list은 카탈로그 선박명 전부에 en_phon을 적용해 리스트 하나
print(catalog_vessel_name_list[0:100])

train_vessel_name_list = []

#훈련용 데이터 
ID_list = []
for i in t_df[train_col['VESSEL_NAME']]:

    name = "".join(normalize_name(i))
    if re.search(r'[가-힣]',name) == None:
        train_vessel_name_list.append(eng_phon(name))
    else:
        train_vessel_name_list.append(ko_phon(name))

#vessel_name_list은 카탈로그 선박명 전부에 en_phon을 적용해 리스트 하나
print(train_vessel_name_list)



In [ ]:
from src.candidates import find_cadidates
Q = train_vessel_name_list
C = catalog_vessel_name_list
top,top_score = find_cadidates(Q,C,4)

print(top)
print(top_score)